In [ ]:
# 1. Instalamos las librerías necesarias
!pip install -U ultralytics kagglehub -q

import kagglehub
import os
import shutil
from google.colab import drive

# 2. Conectamos Google Drive para los guardados
print(" Conectando Google Drive...")
drive.mount('/content/drive', force_remount=True)

os.makedirs('/content/drive/MyDrive/TFG', exist_ok=True)

# 3. Descargamos el dataset sin necesidad de token
print("⬇ Descargando dataset de Búsqueda y Rescate (res-dataset)...")
dataset_path_cache = kagglehub.dataset_download("wxqwxq321/res-dataset")

# 4. Movemos el dataset a una ruta cómoda para YOLO
ruta_dataset = '/content/rescue-myself'
if os.path.exists(ruta_dataset):
    shutil.rmtree(ruta_dataset)
shutil.copytree(dataset_path_cache, ruta_dataset)

print(f" Dataset listo y copiado en: {ruta_dataset}")

In [ ]:
from pathlib import Path
import yaml

print(" Buscando y parcheando data.yaml...")

# 1. Buscamos el archivo en todo el árbol de directorios
directorio_base = Path('/content/rescue-myself')
archivos_yaml = list(directorio_base.rglob('data.yaml'))

if not archivos_yaml:
    print(" Error: No se ha encontrado el archivo data.yaml")
else:

    ruta_yaml = archivos_yaml[0]
    directorio_dataset = ruta_yaml.parent
    print(f" ¡Encontrado en: {ruta_yaml}!")

    # 2. Leemos la configuración original
    with open(ruta_yaml, 'r') as f:
        datos = yaml.safe_load(f)

    # 3. Parcheamos con las rutas dinámicas absolutas de Colab
    datos['train'] = str(directorio_dataset / 'train' / 'images')
    datos['val']   = str(directorio_dataset / 'val' / 'images')

    # Comprobamos si hay carpeta test
    if (directorio_dataset / 'test' / 'images').exists():
        datos['test'] = str(directorio_dataset / 'test' / 'images')

    # Eliminamos el path raíz si existe para evitar conflictos en YOLO
    if 'path' in datos:
        del datos['path']

    # 4. Guardamos los cambios
    with open(ruta_yaml, 'w') as f:
        yaml.dump(datos, f)


In [ ]:
from ultralytics import YOLO
import shutil
import os

print(" Cargando motor YOLO26 Small...")
modelo = YOLO('yolo26s.pt')

print(" Iniciando entrenamiento SAR...")

resultados = modelo.train(
    data="/content/rescue-myself/rescue-myself/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="/content/entrenamiento_tfg",
    name="yolo26_rescue",

    # Transformaciones
    mosaic=1.0,
    mixup=0.15,
    degrees=15.0,
    scale=0.5,
    hsv_s=0.5,
    hsv_v=0.4,
    fliplr=0.5,

    device="0"
)

print("\n Entrenamiento completado.")

# Guardado automático en Drive
ruta_modelo_entrenado = '/content/entrenamiento_tfg/yolo26_rescue/weights/best.pt'
ruta_destino_drive = '/content/drive/MyDrive/TFG/best_yolo26.pt'

if os.path.exists(ruta_modelo_entrenado):
    shutil.copy(ruta_modelo_entrenado, ruta_destino_drive)
    print(f" Modelo guardado a salvo en: {ruta_destino_drive}")
else:
    print(" Hubo un problema al localizar el archivo best.pt. Revisa la carpeta de ejecución.")

In [ ]:
import shutil
import os

# 1. Actualizamos la ruta a la nueva carpeta que ha creado YOLO
ruta_modelo_entrenado = '/content/entrenamiento_tfg/yolo26_rescue2/weights/best.pt'
ruta_destino_drive = '/content/drive/MyDrive/TFG/best_yolo26.pt'

# 2. Hacemos el copiado al Drive
if os.path.exists(ruta_modelo_entrenado):
    shutil.copy(ruta_modelo_entrenado, ruta_destino_drive)
    print(f"El modelo ha sido guardado a salvo en Drive:")
    print(f"   -> {ruta_destino_drive}")
else:
    print(" Sigue sin encontrarse. ")

In [ ]:
from ultralytics import YOLO

# 1. Cargamos el modelo campeón desde tu Drive
ruta_pt = '/content/drive/MyDrive/TFG/best_yolo26.pt'
print(" Cargando modelo PyTorch original...")
modelo_local = YOLO(ruta_pt)

# 2. Exportamos a formato ONNX optimizado para Edge Computing
print(" Iniciando conversión a ONNX...")

# imgsz=640: Mantiene la resolución con la que hemos entrenado.
# half=True: Reduce los pesos de 32 a 16 bits (FP16). Esto es vital para la web,
# ya que reduce el peso del archivo a la mitad y duplica la velocidad de inferencia
# en dispositivos de bajos recursos sin perder precisión real.
ruta_onnx = modelo_local.export(format='onnx', imgsz=640, half=True)

print("\n ¡Conversión completada con éxito!")
print(f" el modelo web está listo en: {ruta_onnx}")

In [ ]:
from ultralytics import YOLO

ruta_pt = '/content/drive/MyDrive/TFG/best_yolo26.pt'
print(" Cargando modelo PyTorch original...")
modelo_local = YOLO(ruta_pt)

# Exportamos SIN half y CON nms
print(" Iniciando conversión a ONNX (FP32 + NMS)...")
ruta_onnx = modelo_local.export(format='onnx', imgsz=640, nms=True)

print(f"\n ¡Modelo Web Definitivo listo en: {ruta_onnx}")